## Cell 1 - Overview
Simple, standardized 10-fold CV MLP notebook (aligned with LR/RF/XGB protocol).

## Cell 2 - Imports

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

## Cell 3 - Settings

In [ ]:
# Baseline protocol settings
DATASET_FILE = "allbp.tsv.gz"
RANDOM_SEED = 90483257
CV_N_SPLITS = 10
CV_SEED = 90483257

# MLP hyperparameters
HIDDEN_NEURONS = 32
HIDDEN_ACTIVATION = "relu"  # relu, leaky_relu, gelu, tanh, sigmoid
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 0.001

## Cell 4 - Reproducibility and Device

In [ ]:
def set_random_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_random_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Cell 5 - Data Loading

In [ ]:
def resolve_dataset_path(path: str) -> Path:
    p = Path(path)
    if p.exists():
        return p

    candidate = Path("results") / "MLP Testing Final" / "Bottom_Datasets" / Path(path).stem.replace(".tsv", "") / Path(path).name
    if candidate.exists():
        return candidate

    search_root = Path("results") / "MLP Testing Final" / "Bottom_Datasets"
    matches = list(search_root.glob(f"*/{Path(path).name}"))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not resolve dataset path: {path}")


def load_dataset(path: str):
    resolved_path = resolve_dataset_path(path)
    df = pd.read_csv(resolved_path, sep='\t', compression='infer')

    if 'class' in df.columns:
        target_col = 'class'
    elif 'target' in df.columns:
        target_col = 'target'
    else:
        target_col = df.columns[-1]

    X = df.drop(columns=[target_col]).astype(np.float32).values
    y_raw = df[target_col].values

    encoder = LabelEncoder()
    y = encoder.fit_transform(y_raw)

    return X, y, target_col, encoder, resolved_path


X, y, target_col, encoder, resolved_path = load_dataset(DATASET_FILE)
num_features = X.shape[1]
num_classes = len(np.unique(y))

print(f"Dataset: {DATASET_FILE}")
print(f"Resolved path: {resolved_path}")
print(f"Target column: {target_col}")
print(f"Samples: {X.shape[0]}, Features: {num_features}, Classes: {num_classes}")

## Cell 6 - Model Definition

In [ ]:
def get_activation(name: str):
    name = name.lower()
    if name == 'relu':
        return nn.ReLU()
    if name == 'leaky_relu':
        return nn.LeakyReLU(negative_slope=0.01)
    if name == 'gelu':
        return nn.GELU()
    if name == 'tanh':
        return nn.Tanh()
    if name == 'sigmoid':
        return nn.Sigmoid()
    raise ValueError(f"Unsupported activation: {name}")


class TabularMLP(nn.Module):
    def __init__(self, in_features: int, hidden_neurons: int, out_features: int, activation: str):
        super().__init__()
        act = get_activation(activation)
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_neurons),
            act,
            nn.Linear(hidden_neurons, hidden_neurons),
            act,
            nn.Linear(hidden_neurons, out_features),
        )

    def forward(self, x):
        return self.net(x)

## Cell 7 - Fold Training Function

In [ ]:
def fit_one_fold(X_train, y_train, X_test, y_test, fold_idx=None):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    X_test_scaled = scaler.transform(X_test).astype(np.float32)

    X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32, device=device)

    binary = num_classes == 2
    out_dim = 1 if binary else num_classes

    if binary:
        y_train_t = torch.tensor(y_train.astype(np.float32), dtype=torch.float32, device=device).view(-1, 1)
        y_test_t = torch.tensor(y_test.astype(np.float32), dtype=torch.float32, device=device).view(-1, 1)
        criterion = nn.BCEWithLogitsLoss()
    else:
        y_train_t = torch.tensor(y_train.astype(np.int64), dtype=torch.long, device=device)
        y_test_t = torch.tensor(y_test.astype(np.int64), dtype=torch.long, device=device)
        criterion = nn.CrossEntropyLoss()

    model = TabularMLP(num_features, HIDDEN_NEURONS, out_dim, HIDDEN_ACTIVATION).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)

    model.train()
    for epoch in range(1, EPOCHS + 1):
        running_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * batch_X.size(0)

        if epoch == 1 or epoch % 25 == 0 or epoch == EPOCHS:
            avg_loss = running_loss / len(train_loader.dataset)
            prefix = f"[Fold {fold_idx:02d}] " if fold_idx is not None else ""
            print(f"{prefix}Epoch {epoch:4d}/{EPOCHS} - Loss: {avg_loss:.6f}")

    model.eval()
    with torch.no_grad():
        logits = model(X_test_t)
        if binary:
            probs = torch.sigmoid(logits).squeeze(1)
            y_pred = (probs >= 0.5).cpu().numpy().astype(int)
        else:
            y_pred = torch.argmax(logits, dim=1).cpu().numpy().astype(int)

    y_true = y_test.astype(int)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    return y_true, y_pred, acc, bal_acc, macro_f1

## Cell 8 - 10-Fold CV Execution

In [ ]:
cv = StratifiedKFold(n_splits=CV_N_SPLITS, shuffle=True, random_state=CV_SEED)

oof_true = []
oof_pred = []
fold_metrics = []

for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    y_true, y_pred, acc, bal_acc, macro_f1 = fit_one_fold(
        X_train, y_train, X_test, y_test, fold_idx=fold_idx
    )

    oof_true.extend(y_true.tolist())
    oof_pred.extend(y_pred.tolist())
    fold_metrics.append({
        'fold': fold_idx,
        'accuracy': acc,
        'balanced_accuracy': bal_acc,
        'macro_f1': macro_f1,
        'train_size': len(train_idx),
        'test_size': len(test_idx),
    })

    print(f"Fold {fold_idx:02d}: acc={acc:.4f}, bal_acc={bal_acc:.4f}, macro_f1={macro_f1:.4f}, test_n={len(test_idx)}")

## Cell 9 - Summary, Export, and Plot

In [ ]:
oof_true = np.array(oof_true, dtype=int)
oof_pred = np.array(oof_pred, dtype=int)

cv_accuracy = accuracy_score(oof_true, oof_pred)
cv_balanced_accuracy = balanced_accuracy_score(oof_true, oof_pred)
cv_macro_f1 = f1_score(oof_true, oof_pred, average='macro', zero_division=0)

fold_df = pd.DataFrame(fold_metrics)
summary = {
    'dataset_file': DATASET_FILE,
    'samples': int(X.shape[0]),
    'features': int(num_features),
    'classes': int(num_classes),
    'cv_splits': int(CV_N_SPLITS),
    'seed': int(RANDOM_SEED),
    'oof_accuracy': float(cv_accuracy),
    'oof_balanced_accuracy': float(cv_balanced_accuracy),
    'oof_macro_f1': float(cv_macro_f1),
    'fold_accuracy_mean': float(fold_df['accuracy'].mean()),
    'fold_accuracy_std': float(fold_df['accuracy'].std(ddof=0)),
    'fold_balanced_accuracy_mean': float(fold_df['balanced_accuracy'].mean()),
    'fold_balanced_accuracy_std': float(fold_df['balanced_accuracy'].std(ddof=0)),
    'fold_macro_f1_mean': float(fold_df['macro_f1'].mean()),
    'fold_macro_f1_std': float(fold_df['macro_f1'].std(ddof=0)),
}

print("\n" + "=" * 72)
print("FINAL 10-FOLD CV RESULTS (OOF PREDICTIONS)")
print("=" * 72)
print(f"Accuracy:           {summary['oof_accuracy']:.4f}")
print(f"Balanced Accuracy:  {summary['oof_balanced_accuracy']:.4f}")
print(f"Macro F1:           {summary['oof_macro_f1']:.4f}")
print("\nFold means +/- std:")
print(f"Accuracy:           {summary['fold_accuracy_mean']:.4f} +/- {summary['fold_accuracy_std']:.4f}")
print(f"Balanced Accuracy:  {summary['fold_balanced_accuracy_mean']:.4f} +/- {summary['fold_balanced_accuracy_std']:.4f}")
print(f"Macro F1:           {summary['fold_macro_f1_mean']:.4f} +/- {summary['fold_macro_f1_std']:.4f}")
print("=" * 72)

pd.DataFrame([summary]).to_csv('cv_summary.csv', index=False)
fold_df.to_csv('cv_fold_metrics.csv', index=False)
print('Saved: cv_summary.csv and cv_fold_metrics.csv')

plt.figure(figsize=(8, 5))
plt.plot(fold_df['fold'], fold_df['accuracy'], marker='o', label='Accuracy')
plt.plot(fold_df['fold'], fold_df['balanced_accuracy'], marker='o', label='Balanced Accuracy')
plt.plot(fold_df['fold'], fold_df['macro_f1'], marker='o', label='Macro F1')
plt.xlabel('Fold')
plt.ylabel('Score')
plt.title('Per-fold Metrics')
plt.grid(True)
plt.legend()
plt.show()